DATA CLUSTERING

In [7]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import hdbscan
import umap
from scipy.spatial.distance import cdist

In [8]:
# Load the wrangled database
db = pd.read_csv('db_wrangled.csv')

In [9]:
# Convert data types
bool_cols = ['Alimentare', 'Non Alimentare', 'Tabella Speciale Carburanti', 'Tabella Speciale Farmacie', 'Tabella Speciale Monopolio']
db[bool_cols] = db[bool_cols].astype("boolean")

db["Codice Via"] = db["Codice Via"].astype("Int64")
db["ZD"] = db["ZD"].astype("Int64")

In [10]:
# Count unique values in the specified column
db["Settore Storico Cf Preval"].nunique()

2119

In [11]:
# Count null values in the specified column
db["Settore Storico Cf Preval"].isnull().sum()

np.int64(33)

In [12]:
# Fill null values in the specified column with "missing"
db["Settore Storico Cf Preval"] = db["Settore Storico Cf Preval"].fillna("missing")

In [ ]:
# Perform density-based clustering on text data embeddings for Settore Storico Cf Preval
def process_clustering(df, text_column):    
    # Drop NaNs in the target column to prevent vectorization errors
    df_clean = df.dropna(subset=[text_column]).copy()
    
    # Extract text list
    text_data = df_clean[text_column].astype(str).tolist()

    # Vectorization (SentenceTransformer)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(text_data, show_progress_bar=True)
    
    # Dimensionality Reduction (UMAP)
    umap_reducer = umap.UMAP(
        n_neighbors=15,
        n_components=15,
        min_dist=0.0,
        metric='cosine',
        random_state=42  #Fixed state for reproducibility
    )
    umap_embeddings = umap_reducer.fit_transform(embeddings)
    
    # Clustering (HDBSCAN)
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=10,
        min_samples=5,
        metric='euclidean', #HDBSCAN usually runs on Euclidean distance of the reduced space
        cluster_selection_method='eom',
        prediction_data=True
    )
    cluster_labels = clusterer.fit_predict(umap_embeddings)
    
    # Output Preparation
    df_clean['cluster_label'] = cluster_labels
    
    return df_clean, embeddings

In [14]:
# Name clusters by representative text closest to the cluster center
def name_clusters_by_center(df, embeddings, text_column, label_column='cluster_label'):
    # Create a dictionary to store the new names
    cluster_names = {}
    
    # Get unique labels (excluding noise -1)
    unique_labels = [l for l in df[label_column].unique() if l != -1]
    
    for label in unique_labels:
        # Create a boolean mask for the current cluster
        mask = (df[label_column] == label)
        
        # Extract embeddings for just this cluster
        cluster_vectors = embeddings[mask]
        
        # Calculate the Centroid (Mean Vector)
        centroid = np.mean(cluster_vectors, axis=0).reshape(1, -1)
        
        # Find the row closest to the centroid (Cosine Distance)
        distances = cdist(centroid, cluster_vectors, metric='cosine')
        min_index = np.argmin(distances)
        
        # Get the actual text
        # .iloc[min_index] works here because cluster_vectors and the 
        # df subset are created with the exact same mask, so the order is preserved
        representative_text = df.loc[mask, text_column].iloc[min_index]
        
        cluster_names[label] = representative_text

    # Assign "Noise" to -1
    cluster_names[-1] = "Noise / Outliers"
        
    return cluster_names

In [15]:
# Main Execution
if __name__ == "__main__":
    # Run clustering and get embeddings
    result_df, embeddings = process_clustering(db, "Settore Storico Cf Preval")
    
    # Generate names
    cluster_name_map = name_clusters_by_center(result_df, embeddings, "Settore Storico Cf Preval")
    
    # Apply names to a new column
    result_df['cluster_name'] = result_df['cluster_label'].map(cluster_name_map)
    
    # Save
    result_df.to_csv("db_clustered.csv", index=False)

Batches:   0%|          | 0/755 [00:00<?, ?it/s]

c:\Users\Utente\AppData\Local\Programs\Python\Python312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [16]:
# Drop temporary columns
db = result_df.drop(["cluster_label", "Settore Storico Cf Preval"], axis = 1)

In [17]:
# Rename the cluster_name column
db = db.rename(columns={"cluster_name": "Settore Storico Cf Preval"})

In [18]:
# Count unique labels after clustering
db["Settore Storico Cf Preval"].nunique()

230

In [ ]:
# Save the clustered dataset
db.to_csv("db_clustered.csv", index=False)